In [0]:
# Cell 1 — Create DataFrame + register as SQL table
data = [
    ("RF", "sklearn", 88, 1.2),
    ("XGB", "xgboost", 92, 2.1),
    ("NN", "pytorch", 95, 5.4),
    ("SVM", "sklearn", 78, 0.8),
    ("LR", "sklearn", 82, 0.3)
]
df = spark.createDataFrame(
    data, ["model", "framework", "accuracy", "train_time"]
)
# Register as temp SQL view
df.createOrReplaceTempView("ml_models")

In [0]:
# Cell 2 — Run actual SQL on Spark!
spark.sql("""
    SELECT framework,
           COUNT(*) as model_count,
           ROUND(AVG(accuracy), 2) as avg_accuracy
    FROM ml_models
    GROUP BY framework
    ORDER BY avg_accuracy DESC
""").show()

+---------+-----------+------------+
|framework|model_count|avg_accuracy|
+---------+-----------+------------+
|  pytorch|          1|        95.0|
|  xgboost|          1|        92.0|
|  sklearn|          3|       82.67|
+---------+-----------+------------+



In [0]:
# Cell 3 — Window function in Spark SQL!
spark.sql("""
    SELECT model, framework, accuracy,
           RANK() OVER (
               PARTITION BY framework 
               ORDER BY accuracy DESC
           ) as rank_in_framework
    FROM ml_models
""").show()

+-----+---------+--------+-----------------+
|model|framework|accuracy|rank_in_framework|
+-----+---------+--------+-----------------+
|   NN|  pytorch|      95|                1|
|   RF|  sklearn|      88|                1|
|   LR|  sklearn|      82|                2|
|  SVM|  sklearn|      78|                3|
|  XGB|  xgboost|      92|                1|
+-----+---------+--------+-----------------+



In [0]:
# Cell 4 — Spark SQL vs DataFrame API
# These two are IDENTICAL in performance:

# SQL style:
spark.sql("SELECT * FROM ml_models WHERE accuracy > 85").show()

# DataFrame API style:
from pyspark.sql.functions import col
df.filter(col("accuracy") > 85).show()



+-----+---------+--------+----------+
|model|framework|accuracy|train_time|
+-----+---------+--------+----------+
|   RF|  sklearn|      88|       1.2|
|  XGB|  xgboost|      92|       2.1|
|   NN|  pytorch|      95|       5.4|
+-----+---------+--------+----------+

+-----+---------+--------+----------+
|model|framework|accuracy|train_time|
+-----+---------+--------+----------+
|   RF|  sklearn|      88|       1.2|
|  XGB|  xgboost|      92|       2.1|
|   NN|  pytorch|      95|       5.4|
+-----+---------+--------+----------+

